In [1]:
import pandas as pd

C:\Users\sasan_5omyce5\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
business_json_path = "C:/Users/sasan_5omyce5/Downloads/Yelp-JSON/Yelp JSON/yelp_dataset/yelp_dataset/yelp_academic_dataset_business.json"
reviews_json_path = "C:/Users/sasan_5omyce5/Downloads/Yelp-JSON/Yelp JSON/yelp_dataset/yelp_dataset/yelp_academic_dataset_review.json"

In [3]:
df_business = pd.read_json(business_json_path, lines=True)

In [4]:
df_business.shape

(150346, 14)

In [5]:
df_business = df_business.dropna(subset=['categories'])

In [6]:
df_business.shape

(150243, 14)

In [8]:
auto_businesses = df_business[df_business['categories'].str.contains('Auto Repair|Automotive', case=False)]

In [9]:
auto_business_ids = set(auto_businesses['business_id'])

In [10]:
relevant_reviews = []
chunk_size = 100000

for chunk in pd.read_json(reviews_json_path, lines=True, chunksize=chunk_size):
    # Keep only the reviews that match our automotive business IDs
    filtered_chunk = chunk[chunk['business_id'].isin(auto_business_ids)]
    
    # We only need the text and the stars for our MultinomialNB model
    relevant_reviews.append(filtered_chunk[['text', 'stars']])

# Combine all filtered chunks into one final DataFrame
df_final = pd.concat(relevant_reviews, ignore_index=True)

In [11]:
df_final.head(5)

,text,stars
0,Great staff always helps and always nice. Alwa...,5
1,Took my vehicle here for some work a few years...,5
2,I've been to this location many times when I l...,3
3,"First time here and they did a great job, very...",5
4,A GREAT EXPERIENCE!!!!!!!!! I was a completel...,5


In [12]:
df_final.groupby('stars').describe()

text                                                                
        count  unique                                                top freq
stars                                                                        
1       78604   78300  DO NOT PARK HERE!\nthey are too quick to boot ...   18
2       12640   12620  The tires I bought seem alright.  But watch ou...    3
3        9040    9029  The wash was very average and very little atte...    2
4       19549   19520  This Lexus helped my family and I so much.. th...    2
5      119168  118924  Family owned & operated local business - best ...    3

In [13]:
df_final.to_csv("vehicle_service_reviews.csv", index=False)

In [14]:
print("Dropping null values in stars")
df_final = df_final.dropna(subset=['stars'])
sentiments = []
print("Find sentiments in datasets")
for index, row in df_final.iterrows():
    if row['stars'] > 3 :
        sentiments.append("Good")
    elif row['stars'] < 3 :
        sentiments.append("Bad")
    else :
        sentiments.append("Neutral")
print("Assign sentiments to the dataset")
df_final['sentiments'] = sentiments
print("Completed!")
df_final.head(5)

Dropping null values in stars
Find sentiments in datasets
Assign sentiments to the dataset
Completed!


,text,stars,sentiments
0,Great staff always helps and always nice. Alwa...,5,Good
1,Took my vehicle here for some work a few years...,5,Good
2,I've been to this location many times when I l...,3,Neutral
3,"First time here and they did a great job, very...",5,Good
4,A GREAT EXPERIENCE!!!!!!!!! I was a completel...,5,Good


In [15]:
df_final = df_final.drop('stars',axis=1)
df_final.head(5)

,text,sentiments
0,Great staff always helps and always nice. Alwa...,Good
1,Took my vehicle here for some work a few years...,Good
2,I've been to this location many times when I l...,Neutral
3,"First time here and they did a great job, very...",Good
4,A GREAT EXPERIENCE!!!!!!!!! I was a completel...,Good


In [16]:
df_final.groupby('sentiments').describe()

text                                                             \
             count  unique                                                top   
sentiments                                                                      
Bad          91244   90911  DO NOT PARK HERE!\nthey are too quick to boot ...   
Good        138717  138436  Family owned & operated local business - best ...   
Neutral       9040    9029  The wash was very average and very little atte...   

                 
           freq  
sentiments       
Bad          18  
Good          3  
Neutral       2

In [17]:
# Keep only rows where sentiment is 'Good' or 'Bad'
df_final = df_final[df_final['sentiments'].isin(['Good', 'Bad'])].copy()
df_final.groupby('sentiments').describe()

text                                                             \
             count  unique                                                top   
sentiments                                                                      
Bad          91244   90911  DO NOT PARK HERE!\nthey are too quick to boot ...   
Good        138717  138436  Family owned & operated local business - best ...   

                 
           freq  
sentiments       
Bad          18  
Good          3

In [19]:
# check new distribution
print(df_final['sentiments'].value_counts(normalize=True))

sentiments
Good    0.60322
Bad     0.39678
Name: proportion, dtype: float64


In [20]:
df_final.to_csv("vehicle_service_reviews_sentiments.csv", index=False)

In [21]:
import pandas as pd

In [22]:
df = pd.read_csv('vehicle_service_reviews_sentiments.csv')
df.head(5)

,text,sentiments
0,Great staff always helps and always nice. Alwa...,Good
1,Took my vehicle here for some work a few years...,Good
2,"First time here and they did a great job, very...",Good
3,A GREAT EXPERIENCE!!!!!!!!! I was a completel...,Good
4,OMG!!! I can't believe this kind of service ex...,Bad
